In [1]:
# Setup PYTHONPATH for imports
import sys
import os

# Method 1: Add current project root to Python path
project_root = "/Users/thanhtrung/Documents/Zalo Received Files/project/for_fun/god-eyes"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Method 2: Set environment variable (optional)
os.environ['PYTHONPATH'] = project_root

print(f"✅ Added to Python path: {project_root}")
print(f"Current sys.path contains project: {project_root in sys.path}")

✅ Added to Python path: /Users/thanhtrung/Documents/Zalo Received Files/project/for_fun/god-eyes
Current sys.path contains project: True


In [2]:
import asyncio
from dotenv import load_dotenv
from src.base.factory import ModelFactory
from src.config.base_config import BaseModelConfig, Provider, ModelType
from src.config.provider_config import OpenRouterConfig

# Load environment variables
load_dotenv()

# Import providers registry to auto-register all providers
from src.providers import registry

print("✅ All imports successful!")
print("Available models:")
for key, model_class in ModelFactory.list_available_models().items():
    print(f"  {key}: {model_class.__name__}")

✓ OpenRouter provider loaded
✓ OpenAI provider loaded
✓ Anthropic provider loaded
✅ All imports successful!
Available models:
  open_router_text: OpenRouterTextModel
  open_router_multimodal: OpenRouterMultiModalModel
  openai_text: OpenAITextModel
  openai_multimodal: OpenAIVisionModel
  anthropic_text: AnthropicTextModel


In [3]:
# Create VLM model configuration
api_key = os.getenv("OPENROUTER_API_KEY")
vlm_model_name = os.getenv("VLM_MODEL_NAME", "")

print(f"API Key found: {'Yes' if api_key else 'No'}")
print(f"VLM Model: {vlm_model_name}")

if api_key:
    # Create VLM model
    config = BaseModelConfig(
        model_name=vlm_model_name,
        provider=Provider.OPEN_ROUTER,
        model_type=ModelType.MULTIMODAL,
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
        temperature=0.7,
        max_tokens=500
    )
    
    # Debug: Check config values
    print(f"🔍 Debug Config:")
    print(f"  Model Type: {config.model_type}")
    print(f"  Model Type Value: {config.model_type.value}")
    print(f"  Expected: {ModelType.MULTIMODAL}")
    print(f"  Expected Value: {ModelType.MULTIMODAL.value}")
    print(f"  Are Equal: {config.model_type == ModelType.MULTIMODAL}")
    
    # Validate config
    try:
        print(f"  Config validation: {config.validate()}")
    except Exception as e:
        print(f"  Config validation error: {e}")
    
    # Try to create model
    try:
        vlm_model = ModelFactory.create_model(config)
        print(f"✅ Model created: {type(vlm_model).__name__}")
        print(f"✅ Model available: {vlm_model.is_available}")
    except Exception as e:
        print(f"❌ Error creating model: {e}")
        
        # Additional debug: Check what's registered
        print(f"\n🔍 Registered models:")
        for key, model_class in ModelFactory.list_available_models().items():
            print(f"  {key}: {model_class.__name__}")
        
        # Check if the specific combination is registered
        key = f"{Provider.OPEN_ROUTER.value}_{ModelType.MULTIMODAL.value}"
        print(f"\nLooking for key: {key}")
        print(f"Is registered: {ModelFactory.is_registered(Provider.OPEN_ROUTER, ModelType.MULTIMODAL)}")
        
else:
    print("⚠️ No API key found. Please check your .env file.")

API Key found: Yes
VLM Model: nvidia/nemotron-nano-12b-v2-vl:free
🔍 Debug Config:
  Model Type: ModelType.MULTIMODAL
  Model Type Value: multimodal
  Expected: ModelType.MULTIMODAL
  Expected Value: multimodal
  Are Equal: True
  Config validation: True
✅ Model created: OpenRouterMultiModalModel
✅ Model available: True


### IMAGE ANALYZER IMPLEMENT

In [4]:
# Sửa lại function vlm_chat cho notebook
async def vlm_chat(query: str, image_urls):
    """
    VLM chat function for Jupyter notebook (async)
    """
    if not vlm_model.is_available:
        raise RuntimeError("VLM model is not available")

    if not hasattr(vlm_model, 'generate_multimodal'):
        raise RuntimeError("Model doesn't support multimodal generation")

    # Đảm bảo image_urls là list
    if isinstance(image_urls, str):
        image_urls = [image_urls]
    
    response = await getattr(vlm_model, 'generate_multimodal')(
        query,
        images=image_urls
    )
    return response

# Function sync cho notebook (sử dụng await trực tiếp)
def vlm_chat_sync(query: str, image_urls):
    """
    Synchronous wrapper - sử dụng trong notebook cell với await
    """
    if isinstance(image_urls, str):
        image_urls = [image_urls]
    
    return vlm_chat(query, image_urls)

In [5]:
# Test với async/await trong notebook
test_images = "https://imageonline.co/image.jpg"

# Cách 1: Sử dụng await trực tiếp (khuyên dùng trong notebook)
response = await vlm_chat("What is shown in the image?", test_images)
print(f"Response: {response}")

Response: The image shows two large, light purple (with darker purple centers) flowers with a trumpet-like shape, surrounded by green leaves. The flowers have a textured appearance and are likely a type of vine flower, such as moonflowers, with soft, overlapping petals. The background includes more greenery, suggesting a natural, outdoor setting.  
So, the final result is Two large, light purple flowers with darker purple centers and green leaves in the background.



### ImageSearch Implement

In [6]:
# Quick test ImageSearch
from src.core.agent.image_search import ImageSearch
from src.config.model_config import ImageSearchConfig
from src.config.base_config import Provider
from dotenv import load_dotenv

load_dotenv()
# Create config
config = ImageSearchConfig(
    model_name="google_search",
    provider=Provider.GOOGLE,
    api_key=os.getenv("GOOGLE_SEARCH_ENGINE_KEY"),
    search_engine_id=os.getenv("GOOGLE_ID_CSE", "")
)

print(f"API Key: {'Found' if config.api_key else 'Not found'}")
print(f"Search Engine ID: {'Found' if config.search_engine_id else 'Not found'}")
print(f"Config valid: {config.validate()}")

# Test search
searcher = ImageSearch(config)
print(f"Available: {searcher.is_available()}")

if searcher.is_available():
    urls = searcher.search("cat", num=3)
    print(f"Found {len(urls)} images:")
    for url in urls:
        print(f"  - {url}")
else:
    print("❌ Not available - check API credentials")

API Key: Found
Search Engine ID: Found
Config valid: True
Available: True
Found 3 images:
  - https://upload.wikimedia.org/wikipedia/commons/4/4d/Cat_November_2010-1a.jpg
  - https://images.squarespace-cdn.com/content/v1/607f89e638219e13eee71b1e/1684821560422-SD5V37BAG28BURTLIXUQ/michael-sum-LEpfefQf4rU-unsplash.jpg
  - https://i.guim.co.uk/img/media/327aa3f0c3b8e40ab03b4ae80319064e401c6fbc/377_133_3542_2834/master/3542.jpg?width=1200&height=1200&quality=85&auto=format&fit=crop&s=34d32522f47e4a67286f9894fc81c863
